# DNA Methylation Ensemble Stacking Pipeline

This notebook implements a stacked ensemble survival clock. It trains five base survival learners (Random Survival Forest, Gradient Boosting Machine, Fast Kernel SVM, Weibull AFT, and MTLR), computes Out-of-Fold (OOF) predictions for the training set, standard-scales predictions, and trains a DeepSurv-style neural network meta-learner using Cox partial likelihood loss.


### Stacking and Calibration Coupling Notes:
- The input file `./coxnet_selected_cpgs_190.csv` is generated by running the feature-selection notebook (`Feature_selection_cox_github.ipynb`) and renaming/copying the resulting CSV.
- The `RUN_ID` value in the stacking, age-regressor, and model re-evaluation cells is hardcoded. After executing the first pipeline cell (which trains the base learners), identify the generated run folder name (e.g. `FHS60_YYYYMMDDHHMMSS`) printed in the logs and update `RUN_ID = '...'` in the subsequent cells to match this run.

In [ ]:
import os
import gc
import json
import random
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from joblib import dump, load

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.linear_model import LinearRegression

from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.svm import FastKernelSurvivalSVM

from lifelines import WeibullAFTFitter
from lifelines.exceptions import ConvergenceError

import tensorflow as tf

# Optional MTLR (pycox/torch) dependencies
try:
    from pycox.models import MTLR as PycoxMTLR
    from torchtuples.practical import MLPVanilla
    from torchtuples import optim as ttoptim
    from torchtuples.callbacks import EarlyStopping
    import torch
    HAS_PYCOX = True
except Exception:
    HAS_PYCOX = False

# ==========================================================
# FHS60 SURVIVAL PIPELINE (WITH WHI EXTERNAL, OLD MTLR LOGIC)
# NEW ROOT: ./FHS60_pipeline
# ==========================================================





# Optional MTLR (pycox)


# ==========================================================
# 0. GLOBAL CONFIG & ROOT DIRECTORIES (ISOLATED)
# ==========================================================
SEED = 42

FHS60_ROOT          = Path(r"./FHS60_pipeline")
FHS60_SPLIT_DIR     = FHS60_ROOT / "splits"
FHS60_SCALER_DIR    = FHS60_ROOT / "scalers"
FHS60_PRED_DIR      = FHS60_ROOT / "predictions"          # (no stacking_oof)
FHS60_MODEL_DIR     = FHS60_ROOT / "savedmodels"
FHS60_LOG_DIR       = FHS60_ROOT / "logs"

# NEW: store per-model prediction scalers (like old project)
FHS60_PRED_SCALERS  = FHS60_ROOT / "pred_scalers"

for d in [FHS60_SPLIT_DIR, FHS60_SCALER_DIR, FHS60_PRED_DIR, FHS60_MODEL_DIR, FHS60_LOG_DIR, FHS60_PRED_SCALERS]:
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = dt.datetime.now().strftime("FHS60_%Y%m%d%H%M%S")
print(f"[INFO] RUN_ID = {RUN_ID}")


# ==========================================================
# 1. LOAD FHS & WHI FULL TABLES
# ==========================================================
print("[INFO] Loading FHS and WHI parquet files...")
FHS = pd.read_parquet(
    r"./data/FHS_methylation_merged.parquet",
    engine="pyarrow",
    thrift_string_size_limit=2_000_000_000,
    thrift_container_size_limit=2_000_000_000,
)
WHI = pd.read_parquet(
    r"./data/WHI_full_imputed.parquet",
    engine="pyarrow",
    thrift_string_size_limit=2_000_000_000,
    thrift_container_size_limit=2_000_000_000,
)
print("[INFO] FHS shape:", FHS.shape)
print("[INFO] WHI shape:", WHI.shape)

FHS.columns = FHS.columns.astype(str).str.strip()
WHI.columns = WHI.columns.astype(str).str.strip()


# ==========================================================
# 2. LOAD ELASTIC-NET CpGs
# ==========================================================
features = pd.read_csv(
    r"./coxnet_selected_cpgs_190.csv",
    low_memory=False,
)
cpg_list = features["CpG"].astype(str).str.strip().dropna().unique().tolist()
print(f"[INFO] Elastic-net CpGs: {len(cpg_list)}")

fhs_meta = ["Patient_ID", "sex", "age", "event", "batch", "time"]
whi_meta = ["Subject_ID", "age", "sex", "event", "time"]

for col in fhs_meta:
    if col not in FHS.columns:
        raise ValueError(f"[ERROR] FHS missing metadata column '{col}'")
for col in whi_meta:
    if col not in WHI.columns:
        raise ValueError(f"[ERROR] WHI missing metadata column '{col}'")

fhs_cpg_keep = [c for c in cpg_list if c in FHS.columns]
whi_cpg_keep = [c for c in cpg_list if c in WHI.columns]
print(f"[INFO] CpGs in FHS: {len(fhs_cpg_keep)}, in WHI: {len(whi_cpg_keep)}")


# ==========================================================
# 3. BUILD FHS_small / WHI_small
# ==========================================================
FHS_small = FHS.loc[:, fhs_meta + fhs_cpg_keep].copy()
WHI_small = WHI.loc[:, whi_meta + whi_cpg_keep].copy()
WHI_small["batch"] = "WHI"

if fhs_cpg_keep:
    FHS_small.loc[:, fhs_cpg_keep] = FHS_small.loc[:, fhs_cpg_keep].astype(np.float32)
if whi_cpg_keep:
    WHI_small.loc[:, whi_cpg_keep] = WHI_small.loc[:, whi_cpg_keep].astype(np.float32)

del FHS, WHI
gc.collect()

print("[INFO] FHS_small:", FHS_small.shape)
print("[INFO] WHI_small:", WHI_small.shape)


# ==========================================================
# 4. FHS 60/20/20 SPLIT + SCALING
# ==========================================================
meta_cols = ["Patient_ID", "sex", "age", "batch", "time", "event"]
for col in meta_cols:
    if col not in FHS_small.columns:
        raise ValueError(f"[ERROR] Required column '{col}' not in FHS_small.")

cpg_cols = [c for c in FHS_small.columns if c not in meta_cols]
print(f"[INFO] CpG feature count: {len(cpg_cols)}")

X_full    = FHS_small[cpg_cols].copy()
y_full    = FHS_small[["time", "event"]].copy()
meta_full = FHS_small[["Patient_ID", "sex", "age", "batch"]].copy()

y_full["event"] = y_full["event"].astype(int)
event_all = y_full["event"].values

train_idx, temp_idx = train_test_split(
    np.arange(len(FHS_small)),
    test_size=0.4,
    stratify=event_all,
    random_state=SEED,
)
temp_events = event_all[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_events,
    random_state=SEED,
)

def describe_split(name, idx_array):
    n = len(idx_array)
    ev = y_full.iloc[idx_array]["event"].values
    n_events = int(ev.sum())
    n_cens = n - n_events
    erate = n_events / n
    print(f"{name} N={n}, events={n_events}, censored={n_cens}, event_rate={erate:.3f}")

print("=== FHS splits (60/20/20, stratified by event) ===")
describe_split("Train", train_idx)
describe_split("Val",   val_idx)
describe_split("Test",  test_idx)

X_train = X_full.iloc[train_idx].copy()
X_val   = X_full.iloc[val_idx].copy()
X_test  = X_full.iloc[test_idx].copy()

y_train = y_full.iloc[train_idx].copy()
y_val   = y_full.iloc[val_idx].copy()
y_test  = y_full.iloc[test_idx].copy()

meta_train = meta_full.iloc[train_idx].copy()
meta_val   = meta_full.iloc[val_idx].copy()
meta_test  = meta_full.iloc[test_idx].copy()

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

print("[INFO] Fitting StandardScaler on FHS TRAIN CpGs only...")
cpg_scaler = SkScaler()
X_train_scaled = cpg_scaler.fit_transform(X_train)
X_val_scaled   = cpg_scaler.transform(X_val)
X_test_scaled  = cpg_scaler.transform(X_test)

scaler_path = FHS60_SCALER_DIR / f"FHS60_train_cpg_scaler_{RUN_ID}.joblib"
dump(cpg_scaler, scaler_path)
print(f"[INFO] Saved CpG scaler: {scaler_path}")

# apply same CpG list + scaler to WHI for external evaluation (NOT used in training)
X_whi = WHI_small[whi_cpg_keep].values.astype(np.float32)
X_whi_scaled = cpg_scaler.transform(X_whi)
y_whi = WHI_small[["time", "event"]].copy()
y_whi["event"] = y_whi["event"].astype(int)

# optional: save raw splits
train_df = pd.concat([meta_train.reset_index(drop=True), y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
val_df   = pd.concat([meta_val.reset_index(drop=True),   y_val.reset_index(drop=True),   X_val.reset_index(drop=True)], axis=1)
test_df  = pd.concat([meta_test.reset_index(drop=True),  y_test.reset_index(drop=True),  X_test.reset_index(drop=True)], axis=1)

train_csv_path = FHS60_SPLIT_DIR / f"FHS60_train_raw_{RUN_ID}.csv"
val_csv_path   = FHS60_SPLIT_DIR / f"FHS60_val_raw_{RUN_ID}.csv"
test_csv_path  = FHS60_SPLIT_DIR / f"FHS60_test_raw_{RUN_ID}.csv"
train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)
print("[INFO] Saved FHS60 raw split CSVs.")

gc.collect()


In [ ]:
# ==========================================================
# 5. SURV ARRAYS + C-INDEX
# ==========================================================
def make_surv_array(y_df):
    return Surv.from_dataframe("event", "time", y_df[["event", "time"]])

y_train_struct = make_surv_array(y_train)
y_val_struct   = make_surv_array(y_val)
y_test_struct  = make_surv_array(y_test)
y_whi_struct   = make_surv_array(y_whi)

def cindex(y_struct, scores):
    e = y_struct["event"].astype(bool)
    t = y_struct["time"].astype(float)
    s = np.asarray(scores).ravel()
    return float(concordance_index_censored(e, t, s)[0])


# ==========================================================
# 6. WEIBULL AFT ADAPTER
# ==========================================================
class WeibullAFTAdapter:
    def __init__(self, penalizer=0.1, l1_ratio=0.0):
        self.penalizer = float(penalizer)
        self.l1_ratio = float(l1_ratio)
        self.model = None

    def get_params(self, deep=True):
        return {"penalizer": self.penalizer, "l1_ratio": self.l1_ratio}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def fit(self, X, y_struct):
        X = np.asarray(X)
        cols = [f"x{i}" for i in range(X.shape[1])]
        Xdf = pd.DataFrame(X, columns=cols)
        ydf = pd.DataFrame(
            {
                "time":  y_struct["time"].astype(float),
                "event": y_struct["event"].astype(bool).astype(int),
            }
        )
        df_train = pd.concat([Xdf.reset_index(drop=True), ydf.reset_index(drop=True)], axis=1)

        self.model = WeibullAFTFitter(penalizer=self.penalizer, l1_ratio=self.l1_ratio)
        try:
            self.model.fit(df_train, duration_col="time", event_col="event")
        except ConvergenceError as e:
            raise RuntimeError(f"WeibullAFT convergence failed: {e}")
        return self

    def predict(self, X):
        X = np.asarray(X)
        cols = [f"x{i}" for i in range(X.shape[1])]
        Xdf = pd.DataFrame(X, columns=cols)
        exptime = self.model.predict_expectation(Xdf)
        return -np.asarray(exptime).ravel()


# ==========================================================
# 7. BASE MODEL PARAMS & CONSTRUCTORS
# ==========================================================
best_params = {
    "rsf": dict(n_estimators=450, max_depth=6, min_samples_split=30, min_samples_leaf=75, max_features=0.01),
    "gbm": dict(n_estimators=400, learning_rate=0.035, max_depth=3, min_samples_split=5, min_samples_leaf=100, subsample=0.6),
    "svm": dict(alpha=0.001554726368159204, gamma=5.2631578947368424e-05, max_iter=40000, tol=0.0001, rank_ratio=1.0),
    "weibull": dict(penalizer=0.01, l1_ratio=0.3),
}

def rsf_ctor(params):
    return RandomSurvivalForest(bootstrap=True, oob_score=False, n_jobs=-1, random_state=SEED, **params)

def gbm_ctor(params):
    return GradientBoostingSurvivalAnalysis(random_state=SEED, **params)

def svm_ctor(params):
    return FastKernelSurvivalSVM(kernel="rbf", random_state=SEED, **params)

def weib_ctor(params):
    return WeibullAFTAdapter(**params)

base_model_ctors = {"rsf": rsf_ctor, "gbm": gbm_ctor, "svm": svm_ctor, "weibull": weib_ctor}

def pred_risk(model, X):
    X = np.asarray(X)
    if hasattr(model, "predict"):
        return np.asarray(model.predict(X)).ravel()
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X)).ravel()
    raise RuntimeError(f"No usable predict method for {type(model)}")


# ==========================================================
# 8. FIT FINAL MODELS + GENERATE OOF PREDICTIONS
# ==========================================================
print("\n=== Generating OOF predictions (FHS60 Train) + training final models ===")

pred_train, pred_val, pred_test, pred_whi = {}, {}, {}, {}

FHS60_BASE_MODEL_DIR = FHS60_MODEL_DIR / f"base_models_{RUN_ID}"
FHS60_BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Define 5-fold CV for OOF
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
events_train = y_train["event"].astype(int).values

metrics = {"base_models": {}}
for name, ctor in base_model_ctors.items():
    print(f"\n--- Processing model: {name} ---")
    
    # 1. Fit final model on complete X_train_scaled
    print(f"  Fitting final {name} on FHS60 Train (full)...")
    m_full = ctor(best_params[name])
    m_full.fit(X_train_scaled, y_train_struct)
    
    model_path = FHS60_BASE_MODEL_DIR / f"{name}_full_FHS60_{RUN_ID}.joblib"
    dump(m_full, model_path)
    print(f"  [INFO] Saved final model: {model_path}")
    
    # Generate val, test, whi predictions using the final model
    r_va  = pred_risk(m_full, X_val_scaled)
    r_te  = pred_risk(m_full, X_test_scaled)
    r_whi = pred_risk(m_full, X_whi_scaled)
    
    pred_val[name]   = r_va
    pred_test[name]  = r_te
    pred_whi[name]   = r_whi
    
    # 2. Out-of-fold loop for train set predictions
    print(f"  Generating Out-of-Fold (OOF) train predictions...")
    oof_preds = np.zeros(len(X_train_scaled))
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_scaled, events_train), start=1):
        X_tr_f, X_va_f = X_train_scaled[tr_idx], X_train_scaled[va_idx]
        y_tr_f_struct = Surv.from_dataframe("event", "time", y_train.iloc[tr_idx])
        
        m_fold = ctor(best_params[name])
        m_fold.fit(X_tr_f, y_tr_f_struct)
        oof_preds[va_idx] = pred_risk(m_fold, X_va_f)
        
    pred_train[name] = oof_preds
    
    # Metrics
    m = {
        "train_cindex": cindex(y_train_struct, oof_preds),
        "val_cindex":   cindex(y_val_struct,   r_va),
        "test_cindex":  cindex(y_test_struct,  r_te),
        "whi_cindex":   cindex(y_whi_struct,   r_whi),
        "model_path":   str(model_path),
    }
    metrics["base_models"][name] = m
    
    print(f"  {name} OOF TRAIN C-index = {m['train_cindex']:.4f}")
    print(f"  {name} VAL   C-index = {m['val_cindex']:.4f}")
    print(f"  {name} TEST  C-index = {m['test_cindex']:.4f}")
    print(f"  {name} WHI   C-index = {m['whi_cindex']:.4f}")


# ==========================================================
# 9. MTLR (CPU-only + reproducible) + SAVE BUNDLE WITH OOF
# ==========================================================
INCLUDE_MTLR = HAS_PYCOX
if INCLUDE_MTLR:

    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    os.environ["PYTHONHASHSEED"] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.set_num_threads(1)

    print("\n=== MTLR full-train + OOF (CPU-only) ===")

    cfg = dict(
        num_durations=35,
        layers=(32, 16),
        dropout=0.35,
        batchnorm=False,
        lr=3e-4,
        batch_size=32,
        epochs=400,
        patience=15,
    )
    print("[MTLR] Config:", cfg)

    dur_tr = y_train["time"].astype(float).values
    evt_tr = y_train["event"].astype(bool).values
    dur_va = y_val["time"].astype(float).values
    evt_va = y_val["event"].astype(bool).values

    def mtlr_risk_scores(model, X_np, cuts):
        surv_df = model.predict_surv_df(X_np)
        return (-np.trapz(surv_df.values, x=cuts, axis=0)).astype(np.float64)

    def mtlr_fit_one(cfg, X_tr, dur_tr, evt_tr, X_va, dur_va, evt_va):
        labtrans = PycoxMTLR.label_transform(cfg["num_durations"])
        y_trmt = labtrans.fit_transform(dur_tr, evt_tr)
        y_vamt = labtrans.transform(dur_va, evt_va)
        cuts = labtrans.cuts

        net = MLPVanilla(
            X_tr.shape[1],
            list(cfg["layers"]),
            cfg["num_durations"],
            batch_norm=cfg["batchnorm"],
            dropout=cfg["dropout"],
        )

        opt = ttoptim.Adam(lr=cfg["lr"])
        m = PycoxMTLR(net, opt, duration_index=cuts)

        callbacks = [EarlyStopping(patience=cfg["patience"])]
        m.fit(
            X_tr, y_trmt,
            batch_size=cfg["batch_size"],
            epochs=cfg["epochs"],
            callbacks=callbacks,
            val_data=(X_va, y_vamt),
            verbose=False,
            shuffle=False,
        )
        return m, cuts

    # 1. Fit final MTLR model on complete train set
    print("  Fitting final MTLR model on FHS60 Train (full)...")
    mtlr_model, mtlr_cuts = mtlr_fit_one(
        cfg, X_train_scaled, dur_tr, evt_tr, X_val_scaled, dur_va, evt_va
    )

    r_va  = mtlr_risk_scores(mtlr_model, X_val_scaled,   mtlr_cuts)
    r_te  = mtlr_risk_scores(mtlr_model, X_test_scaled,  mtlr_cuts)
    r_whi = mtlr_risk_scores(mtlr_model, X_whi_scaled,   mtlr_cuts)

    pred_val["mtlr"]   = r_va
    pred_test["mtlr"]  = r_te
    pred_whi["mtlr"]   = r_whi

    mtlr_bundle_path = FHS60_BASE_MODEL_DIR / f"mtlr_full_FHS60_{RUN_ID}.pt"
    torch.save(
        {"state_dict": mtlr_model.net.state_dict(), "cuts": mtlr_cuts, "cfg": cfg},
        mtlr_bundle_path,
    )
    print(f"  [INFO] Saved MTLR bundle: {mtlr_bundle_path}")

    # 2. Out-of-fold loop for training split predictions
    print("  Generating MTLR OOF predictions...")
    oof_mtlr = np.zeros(len(X_train_scaled))
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_scaled, events_train), start=1):
        X_tr_f, X_va_f = X_train_scaled[tr_idx], X_train_scaled[va_idx]
        dtr_f, etr_f = dur_tr[tr_idx], evt_tr[tr_idx]
        dva_f, eva_f = dur_tr[va_idx], evt_tr[va_idx]
        
        m_fold, cuts_fold = mtlr_fit_one(
            cfg, X_tr_f, dtr_f, etr_f, X_va_f, dva_f, eva_f
        )
        oof_mtlr[va_idx] = mtlr_risk_scores(m_fold, X_va_f, cuts_fold)
        
    pred_train["mtlr"] = oof_mtlr

    metrics["mtlr"] = {
        "train_cindex": cindex(y_train_struct, oof_mtlr),
        "val_cindex":   cindex(y_val_struct,   r_va),
        "test_cindex":  cindex(y_test_struct,  r_te),
        "whi_cindex":   cindex(y_whi_struct,   r_whi),
        "bundle_path":  str(mtlr_bundle_path),
        "cfg":          cfg,
    }

    print(f"  MTLR OOF TRAIN C-index = {metrics['mtlr']['train_cindex']:.4f}")
    print(f"  MTLR VAL   C-index = {metrics['mtlr']['val_cindex']:.4f}")
    print(f"  MTLR TEST  C-index = {metrics['mtlr']['test_cindex']:.4f}")
    print(f"  MTLR WHI   C-index = {metrics['mtlr']['whi_cindex']:.4f}")

else:
    print("[INFO] MTLR not available; skipping MTLR model.")


# ==========================================================
# 10. SCALE BASE PREDICTIONS + SAVE STACKING CSVs (RAW + SCALED)
# ==========================================================
model_names = list(pred_train.keys())  # rsf, gbm, svm, weibull, (optional) mtlr

pred_train_scaled = {}
pred_val_scaled   = {}
pred_test_scaled  = {}
pred_whi_scaled   = {}

print("\n=== Scaling base-model preds (fit scaler on FHS60 TRAIN preds only) ===")
for name in model_names:
    sc = SkScaler()
    tr = np.asarray(pred_train[name]).reshape(-1, 1)
    sc.fit(tr)

    pred_train_scaled[name] = sc.transform(tr).ravel()
    pred_val_scaled[name]   = sc.transform(np.asarray(pred_val[name]).reshape(-1, 1)).ravel()
    pred_test_scaled[name]  = sc.transform(np.asarray(pred_test[name]).reshape(-1, 1)).ravel()
    pred_whi_scaled[name]   = sc.transform(np.asarray(pred_whi[name]).reshape(-1, 1)).ravel()

    sc_path = FHS60_PRED_SCALERS / f"pred_scaler_{name}_{RUN_ID}.joblib"
    dump(sc, sc_path)
    print(f"  {name} pred scaler saved: {sc_path}")

def build_pred_df(meta_df, y_df, raw_dict, scaled_dict):
    out = meta_df.reset_index(drop=True).copy()
    out["time"]  = y_df["time"].astype(float).values
    out["event"] = y_df["event"].astype(int).values
    for name in model_names:
        out[f"pred_{name}"]        = np.asarray(raw_dict[name]).ravel()
        out[f"pred_{name}_scaled"] = np.asarray(scaled_dict[name]).ravel()
    return out

train_pred_df = build_pred_df(meta_train, y_train, pred_train, pred_train_scaled)
val_pred_df   = build_pred_df(meta_val,   y_val,   pred_val,   pred_val_scaled)
test_pred_df  = build_pred_df(meta_test,  y_test,  pred_test,  pred_test_scaled)

whi_meta    = WHI_small[["Subject_ID", "age", "sex", "batch"]]
whi_pred_df = build_pred_df(whi_meta, y_whi, pred_whi, pred_whi_scaled)

FHS60_RUN_DIR = FHS60_PRED_DIR / f"FHS60_run_{RUN_ID}"
FHS60_RUN_DIR.mkdir(parents=True, exist_ok=True)

train_pred_path = FHS60_RUN_DIR / "FHS60_preds_train.csv"
val_pred_path   = FHS60_RUN_DIR / "FHS60_preds_val.csv"
test_pred_path  = FHS60_RUN_DIR / "FHS60_preds_test.csv"
whi_pred_path   = FHS60_RUN_DIR / "WHI_preds_external.csv"

train_pred_df.to_csv(train_pred_path, index=False)
val_pred_df.to_csv(val_pred_path,   index=False)
test_pred_df.to_csv(test_pred_path, index=False)
whi_pred_df.to_csv(whi_pred_path,   index=False)

metrics_path = FHS60_RUN_DIR / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n[INFO] Saved prediction CSVs:")
print(" Train:", train_pred_path)
print(" Val:  ", val_pred_path)
print(" Test: ", test_pred_path)
print(" WHI:  ", whi_pred_path)
print("[INFO] Saved metrics:", metrics_path)

gc.collect()
print("[DONE] Full-train (OOF-supported) base-model + MTLR finished.")


In [ ]:
# ==========================================================
# FHS60 STACKING (your pipeline): TensorFlow CoxMLP + DeepSurv Cox PH Loss
#   - Loads scaled OOF base-model predictions (FHS60 + WHI external)
#   - Trains a meta-MLP using Cox partial likelihood (DeepSurv style)
#   - Early stopping on FHS60 validation Cox loss
#   - Reports C-index Train/Val/Test/WHI
# ==========================================================



# ==========================================================
# 0. CONFIG + PATHS (Set RUN_ID to match the folder generated by the pipeline)
# ==========================================================
CFG = {
    "hidden":    (128, 64, 128),
    "dropout":   0.35,
    "lr":        2e-4,
    "epochs":    400,
    "patience":  15,
    "min_delta": 1e-3,
    "seed":      42,
}

# CFG = {
#     "hidden":    (26, 29),
#     "dropout":   0.4761717096463864,
#     "lr":        0.0019949066653183914,
#     "epochs":    400,
#     "patience":  15,
#     "min_delta": 9.66137083067618e-05,
#     "seed":      42,
# }

# Set to the RUN_ID folder produced by the pipeline
RUN_ID = "FHS60_20260119171709"

tf.random.set_seed(CFG["seed"])
np.random.seed(CFG["seed"])

# Root for THIS project (derived from the pipeline outputs)
ROOT = Path(r"./FHS60_pipeline")

OOF_DIR = ROOT / "predictions" / f"FHS60_run_{RUN_ID}"
train_pred_path = OOF_DIR / "FHS60_preds_train.csv"
val_pred_path   = OOF_DIR / "FHS60_preds_val.csv"
test_pred_path  = OOF_DIR / "FHS60_preds_test.csv"
whi_pred_path   = OOF_DIR / "WHI_preds_external.csv"

# Directory where stacking model will be saved
MODEL_DIR = ROOT / "savedmodels" / f"stacking_FHS60_{RUN_ID}"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Loading base-model prediction CSVs...")
trainpreddf = pd.read_csv(train_pred_path)
valpreddf   = pd.read_csv(val_pred_path)
testpreddf  = pd.read_csv(test_pred_path)
whipreddf   = pd.read_csv(whi_pred_path)
print("[INFO] Shapes:", trainpreddf.shape, valpreddf.shape, testpreddf.shape, whipreddf.shape)

# ==========================================================
# 1. SURVIVAL TARGETS + SANITY CHECKS
# ==========================================================
def make_surv_from_df(df: pd.DataFrame):
    if not {"event", "time"}.issubset(df.columns):
        raise ValueError("Missing 'event' or 'time' columns")
    if df["event"].isna().any() or df["time"].isna().any():
        raise ValueError("NaNs in 'event' or 'time'")
    if (df["time"] < 0).any():
        raise ValueError("Negative times present")
    return Surv.from_dataframe("event", "time", df[["event", "time"]])

y_train_struct = make_surv_from_df(trainpreddf)
y_val_struct   = make_surv_from_df(valpreddf)
y_test_struct  = make_surv_from_df(testpreddf)
y_whi_struct   = make_surv_from_df(whipreddf)

# ==========================================================
# 2. META-FEATURE MATRIX FROM SCALED PREDICTIONS
# ==========================================================
metacols = [c for c in trainpreddf.columns if c.startswith("pred_") and c.endswith("_scaled")]
if len(metacols) == 0:
    raise ValueError("No meta-features found. Expected columns like 'pred_*_scaled'")

for df_name, df in [("val", valpreddf), ("test", testpreddf), ("whi", whipreddf)]:
    missing = [c for c in metacols if c not in df.columns]
    if missing:
        raise ValueError(f"[ERROR] {df_name} df missing meta-features: {missing}")

print("[INFO] Meta features:", metacols)

X_meta_train = trainpreddf[metacols].to_numpy(dtype=np.float32)
X_meta_val   = valpreddf[metacols].to_numpy(dtype=np.float32)
X_meta_test  = testpreddf[metacols].to_numpy(dtype=np.float32)
X_meta_whi   = whipreddf[metacols].to_numpy(dtype=np.float32)

t_train = trainpreddf["time"].to_numpy(dtype=np.float32)
e_train = trainpreddf["event"].to_numpy(dtype=np.float32)

t_val   = valpreddf["time"].to_numpy(dtype=np.float32)
e_val   = valpreddf["event"].to_numpy(dtype=np.float32)

t_test  = testpreddf["time"].to_numpy(dtype=np.float32)
e_test  = testpreddf["event"].to_numpy(dtype=np.float32)

t_whi   = whipreddf["time"].to_numpy(dtype=np.float32)
e_whi   = whipreddf["event"].to_numpy(dtype=np.float32)

print("[INFO] X_meta_train shape:", X_meta_train.shape)

# ==========================================================
# 3. DeepSurv Cox PH loss (Breslow-style via cumsum risk set)
# ==========================================================
@tf.function
def deepsurv_cox_loss(event, time, risk):
    event = tf.reshape(tf.cast(event, tf.float32), [-1])
    time  = tf.reshape(tf.cast(time,  tf.float32), [-1])
    risk  = tf.reshape(tf.cast(risk,  tf.float32), [-1])

    # sort by time ascending; cumulative sum in reverse makes risk sets
    order = tf.argsort(time, direction="ASCENDING")
    event = tf.gather(event, order)
    risk  = tf.gather(risk,  order)

    risk = risk - tf.reduce_max(risk)
    exp_risk = tf.exp(risk)
    log_risk_set_sum = tf.math.log(tf.math.cumsum(exp_risk, reverse=True) + 1e-12)

    per_sample = event * (risk - log_risk_set_sum)
    n_events = tf.reduce_sum(event)
    n_events = tf.maximum(n_events, 1.0)

    return -tf.reduce_sum(per_sample) / n_events

# ==========================================================
# 4. CoxMLP meta-model (tf.keras)
# ==========================================================
def build_coxmlp(in_dim, hidden=(128, 64, 32), dropout=0.05):
    inputs = tf.keras.Input(shape=(in_dim,), dtype=tf.float32)
    x = inputs
    for h in hidden:
        x = tf.keras.layers.Dense(h, activation="relu")(x)
        x = tf.keras.layers.Dropout(dropout)(x)
    risk = tf.keras.layers.Dense(1, activation=None)(x)
    risk = tf.keras.layers.Lambda(lambda z: tf.reshape(z, [-1]))(risk)
    return tf.keras.Model(inputs=inputs, outputs=risk, name="CoxMLP")

# ==========================================================
# 5. Custom training wrapper (full-batch)
# ==========================================================
class DeepSurvTrainer(tf.keras.Model):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def call(self, x, training=False):
        return self.base_model(x, training=training)

    def train_step(self, data):
        x, y = data
        event, time = y
        with tf.GradientTape() as tape:
            risk = self(x, training=True)
            loss = deepsurv_cox_loss(event, time, risk)
            if self.losses:
                loss += tf.add_n(self.losses)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {"loss": loss}

    def test_step(self, data):
        x, y = data
        event, time = y
        risk = self(x, training=False)
        loss = deepsurv_cox_loss(event, time, risk)
        return {"loss": loss}

# ==========================================================
# 6. C-index helper (sksurv expects higher score = higher risk)
# ==========================================================
def cindex_sksurv(y_struct, scores):
    e = y_struct["event"].astype(bool)
    t = y_struct["time"].astype(float)
    s = np.asarray(scores).ravel()
    return float(concordance_index_censored(e, t, s)[0])

# ==========================================================
# 7. TRAINING SETUP (full-batch, early stopping on val loss)
# ==========================================================
base = build_coxmlp(
    in_dim=X_meta_train.shape[1],
    hidden=CFG["hidden"],
    dropout=CFG["dropout"],
)
model = DeepSurvTrainer(base)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=CFG["lr"]))

EPOCHS    = CFG["epochs"]
PATIENCE  = CFG["patience"]
MIN_DELTA = CFG["min_delta"]

best_val   = np.inf
best_epoch = None
pat        = 0

best_ckpt_path = MODEL_DIR / f"stacking_deepsurv_best_{RUN_ID}.keras"

n_train = X_meta_train.shape[0]
n_val   = X_meta_val.shape[0]
n_test  = X_meta_test.shape[0]
n_whi   = X_meta_whi.shape[0]

train_ds = tf.data.Dataset.from_tensor_slices((X_meta_train, (e_train, t_train))).batch(n_train)
val_ds   = tf.data.Dataset.from_tensor_slices((X_meta_val,   (e_val,   t_val))).batch(n_val)
test_ds  = tf.data.Dataset.from_tensor_slices((X_meta_test,  (e_test,  t_test))).batch(n_test)
whi_ds   = tf.data.Dataset.from_tensor_slices((X_meta_whi,   (e_whi,   t_whi))).batch(n_whi)

print("[INFO] Starting stacking NN training (full-batch Cox loss)...")

for epoch in range(1, EPOCHS + 1):
    hist = model.fit(train_ds, epochs=1, verbose=0)
    train_loss = float(hist.history["loss"][0])

    val_loss = float(model.evaluate(val_ds, verbose=0, return_dict=True)["loss"])

    rs_train = model.predict(X_meta_train, batch_size=n_train, verbose=0)
    rs_val   = model.predict(X_meta_val,   batch_size=n_val,   verbose=0)
    rs_test  = model.predict(X_meta_test,  batch_size=n_test,  verbose=0)
    rs_whi   = model.predict(X_meta_whi,   batch_size=n_whi,   verbose=0)

    c_train = cindex_sksurv(y_train_struct, rs_train)
    c_val   = cindex_sksurv(y_val_struct,   rs_val)
    c_test  = cindex_sksurv(y_test_struct,  rs_test)
    c_whi   = cindex_sksurv(y_whi_struct,   rs_whi)

    improved = val_loss < (best_val - MIN_DELTA)
    if improved:
        best_val   = val_loss
        best_epoch = epoch
        pat        = 0

        # Save whole model (recommended). Keras saves architecture+weights.
        base.save(best_ckpt_path, include_optimizer=False)

        meta_path = MODEL_DIR / f"stacking_deepsurv_best_meta_{RUN_ID}.json"
        meta = {
            "epoch":         int(epoch),
            "val_loss":      float(best_val),
            "train_cindex":  float(c_train),
            "val_cindex":    float(c_val),
            "test_cindex":   float(c_test),
            "whi_cindex":    float(c_whi),
            "meta_features": metacols,
            "hidden":        list(CFG["hidden"]),
            "dropout":       float(CFG["dropout"]),
            "lr":            float(CFG["lr"]),
            "run_id":        RUN_ID,
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        print(f" -> New best val_loss={best_val:.6f} at epoch {epoch}, saved={best_ckpt_path}")
    else:
        pat += 1

    print(
        f"Epoch {epoch:03d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
        f"C-train={c_train:.4f} | C-val={c_val:.4f} | C-test={c_test:.4f} | C-WHI={c_whi:.4f}"
    )

    if pat >= PATIENCE:
        print(f"[EARLY STOP] epoch={epoch}, best_val={best_val:.6f} at epoch {best_epoch}")
        break

# ==========================================================
# 8. RELOAD BEST MODEL AND FINAL METRICS
# ==========================================================
best_model = tf.keras.models.load_model(best_ckpt_path, compile=False)

rs_train = best_model.predict(X_meta_train, batch_size=n_train, verbose=0)
rs_val   = best_model.predict(X_meta_val,   batch_size=n_val,   verbose=0)
rs_test  = best_model.predict(X_meta_test,  batch_size=n_test,  verbose=0)
rs_whi   = best_model.predict(X_meta_whi,   batch_size=n_whi,   verbose=0)

final_c_train = cindex_sksurv(y_train_struct, rs_train)
final_c_val   = cindex_sksurv(y_val_struct,   rs_val)
final_c_test  = cindex_sksurv(y_test_struct,  rs_test)
final_c_whi   = cindex_sksurv(y_whi_struct,   rs_whi)

print(
    f"[FINAL] CoxMLP DeepSurv best_epoch={best_epoch} "
    f"train={final_c_train:.4f}, val={final_c_val:.4f}, "
    f"test={final_c_test:.4f}, WHI={final_c_whi:.4f}"
)

final_stamp = dt.datetime.now().strftime("%Y%m%d%H%M%S")
final_path = MODEL_DIR / f"stacking_deepsurv_final_{RUN_ID}_{final_stamp}.keras"
best_model.save(final_path, include_optimizer=False)
print(f"[SAVE] Final best model saved to: {final_path}")

gc.collect()


In [ ]:

# ---------------- CONFIG (YOUR CURRENT PIPELINE) ----------------
RUN_ID = "FHS60_20260119171709"  # <- change if needed

ROOT = Path(r"./FHS60_pipeline")

# Directory where base learner predictions are stored
PRED_DIR = ROOT / "predictions" / f"FHS60_run_{RUN_ID}"

TRAIN_PRED = PRED_DIR / "FHS60_preds_train.csv"
VAL_PRED   = PRED_DIR / "FHS60_preds_val.csv"
TEST_PRED  = PRED_DIR / "FHS60_preds_test.csv"
WHI_PRED   = PRED_DIR / "WHI_preds_external.csv"

# Directory where the stacking model is saved
STACK_DIR = ROOT / "savedmodels" / f"stacking_FHS60_{RUN_ID}"
STACK_FINAL_PATH = STACK_DIR / f"stacking_deepsurv_best_{RUN_ID}.keras"  # <- produced in your training loop

# Outputs
OUT_DIR = STACK_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

AGE_REG_PATH = OUT_DIR / f"age_regressor_from_stackNN_{RUN_ID}.joblib"
AGE_CSV_PATH = OUT_DIR / f"age_predictions_from_stackNN_{RUN_ID}.csv"

# Optional: load original FHS/WHI parquet just to attach sex_code robustly
FHS_PARQUET = r"./data/FHS_methylation_merged.parquet"
WHI_PARQUET = r"./data/WHI_full_imputed.parquet"

# ---------------- LOAD PREDICTION DATA ----------------
trainpreddf = pd.read_csv(TRAIN_PRED)
valpreddf   = pd.read_csv(VAL_PRED)
testpreddf  = pd.read_csv(TEST_PRED)
whipreddf   = pd.read_csv(WHI_PRED)

# meta-features = scaled base-model predictions
meta_cols = [c for c in trainpreddf.columns if c.startswith("pred_") and c.endswith("_scaled")]
if len(meta_cols) == 0:
    raise ValueError("No meta-features found (expected columns like pred_*_scaled).")

# Ensure meta columns exist in all splits
for name, df in [("val", valpreddf), ("test", testpreddf), ("whi", whipreddf)]:
    missing = [c for c in meta_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} df missing meta-features: {missing}")

X_meta_train = trainpreddf[meta_cols].to_numpy(dtype=np.float32)
X_meta_val   = valpreddf[meta_cols].to_numpy(dtype=np.float32)
X_meta_test  = testpreddf[meta_cols].to_numpy(dtype=np.float32)
X_meta_whi   = whipreddf[meta_cols].to_numpy(dtype=np.float32)

# age / time / event (must exist in these CSVs from your pipeline)
age_train = trainpreddf["age"].astype(float).to_numpy()
age_val   = valpreddf["age"].astype(float).to_numpy()
age_test  = testpreddf["age"].astype(float).to_numpy()
age_whi   = whipreddf["age"].astype(float).to_numpy()

time_train = trainpreddf["time"].astype(float).to_numpy()
time_val   = valpreddf["time"].astype(float).to_numpy()
time_test  = testpreddf["time"].astype(float).to_numpy()
time_whi   = whipreddf["time"].astype(float).to_numpy()

event_train = trainpreddf["event"].astype(int).to_numpy()
event_val   = valpreddf["event"].astype(int).to_numpy()
event_test  = testpreddf["event"].astype(int).to_numpy()
event_whi   = whipreddf["event"].astype(int).to_numpy()

# ---------------- LOAD TRAINED STACKING NN ----------------
# compile=False is standard when loading for inference only (especially with custom losses).
stack_model = tf.keras.models.load_model(str(STACK_FINAL_PATH), compile=False)

def nn_risk(X: np.ndarray) -> np.ndarray:
    # predict returns shape (n, 1) or (n,), normalize to (n,)
    return stack_model.predict(X, batch_size=X.shape[0], verbose=0).ravel()

rs_train = nn_risk(X_meta_train)
rs_val   = nn_risk(X_meta_val)
rs_test  = nn_risk(X_meta_test)
rs_whi   = nn_risk(X_meta_whi)

print("[INFO] Raw stacking NN score ranges:")
print("  Train:", float(rs_train.min()), "to", float(rs_train.max()))
print("  Val  :", float(rs_val.min()),   "to", float(rs_val.max()))
print("  Test :", float(rs_test.min()),  "to", float(rs_test.max()))
print("  WHI  :", float(rs_whi.min()),   "to", float(rs_whi.max()))

# ---------------- FIT AGE REGRESSOR ON TRAIN (RAW NN) ----------------
# This learns: age ≈ alpha + beta * nn_raw
age_reg = LinearRegression()
age_reg.fit(rs_train.reshape(-1, 1), age_train)

print("[INFO] Age regressor fitted on TRAIN:")
print("  beta (coef)     =", float(age_reg.coef_[0]))
print("  alpha (intercept)=", float(age_reg.intercept_))

joblib.dump(age_reg, AGE_REG_PATH)
print("[INFO] Saved age regressor:", AGE_REG_PATH)

# ---------------- APPLY TO ALL SPLITS ----------------
age_pred_train = age_reg.predict(rs_train.reshape(-1, 1))
age_pred_val   = age_reg.predict(rs_val.reshape(-1, 1))
age_pred_test  = age_reg.predict(rs_test.reshape(-1, 1))
age_pred_whi   = age_reg.predict(rs_whi.reshape(-1, 1))

# ---------------- BUILD OUTPUT TABLE ----------------
rows = []

def add_block(df_src, id_col, split, age_true, age_pred, nn_raw, time_vec, event_vec):
    if id_col not in df_src.columns:
        raise ValueError(f"Missing id column '{id_col}' in {split} df.")
    ids = df_src[id_col].to_numpy()
    for pid, a_t, a_p, s, tm, ev in zip(ids, age_true, age_pred, nn_raw, time_vec, event_vec):
        rows.append({
            "split": split,
            "id": pid,
            "time": float(tm),
            "event": int(ev),
            "age_true": float(a_t),
            "nn_raw": float(s),
            "age_pred": float(a_p),
        })

# FHS uses Patient_ID in your saved prediction CSVs
add_block(trainpreddf, "Patient_ID", "Train",      age_train, age_pred_train, rs_train, time_train, event_train)
add_block(valpreddf,   "Patient_ID", "Validation", age_val,   age_pred_val,   rs_val,   time_val,   event_val)
add_block(testpreddf,  "Patient_ID", "Test",       age_test,  age_pred_test,  rs_test,  time_test,  event_test)

# WHI uses Subject_ID
add_block(whipreddf, "Subject_ID", "WHI", age_whi, age_pred_whi, rs_whi, time_whi, event_whi)

age_df = pd.DataFrame(rows)

# ---------------- ATTACH sex_code (ROBUST) ----------------
# We will use:
# - FHS: read parquet and map Patient_ID -> sex_code
# - WHI: set sex_code = 2 (WHI is female-only in your earlier script logic)

FHS = pd.read_parquet(
    FHS_PARQUET,
    engine="pyarrow",
    thrift_string_size_limit=2_000_000_000,
    thrift_container_size_limit=2_000_000_000,
)
WHI = pd.read_parquet(
    WHI_PARQUET,
    engine="pyarrow",
    thrift_string_size_limit=2_000_000_000,
    thrift_container_size_limit=2_000_000_000,
)

FHS.columns = FHS.columns.astype(str).str.strip()
WHI.columns = WHI.columns.astype(str).str.strip()

if "Patient_ID" not in FHS.columns or "sex" not in FHS.columns:
    raise ValueError("FHS parquet must contain Patient_ID and sex columns.")

fhs_sex_map = FHS[["Patient_ID", "sex"]].drop_duplicates(subset="Patient_ID").copy()
if fhs_sex_map["sex"].dtype == object:
    # adjust if your encoding differs
    fhs_sex_map["sex_code"] = fhs_sex_map["sex"].map({"M": 1, "F": 2})
else:
    fhs_sex_map["sex_code"] = fhs_sex_map["sex"].astype(int)

fhs_sex_map = fhs_sex_map[["Patient_ID", "sex_code"]]

# merge for FHS splits
mask_whi = age_df["split"].eq("WHI")
age_df_fhs = age_df.loc[~mask_whi].rename(columns={"id": "Patient_ID"}).merge(
    fhs_sex_map, how="left", on="Patient_ID"
).rename(columns={"Patient_ID": "id"})

# WHI: set sex_code = 2
age_df_whi = age_df.loc[mask_whi].copy()
age_df_whi["sex_code"] = 2

age_df_final = pd.concat([age_df_fhs, age_df_whi], axis=0, ignore_index=True)

if age_df_final["sex_code"].isna().any():
    missing = age_df_final.loc[age_df_final["sex_code"].isna(), "id"].unique()
    print("[WARN] Missing sex_code for some ids:", missing[:10], "(showing up to 10)")

# ---------------- SAVE ----------------
age_df_final.to_csv(AGE_CSV_PATH, index=False)
print("[INFO] Saved age prediction CSV:", AGE_CSV_PATH)

del FHS, WHI
gc.collect()


In [ ]:
# ==========================================================
# LOAD STACKING CoxMLP (.keras) + RECOMPUTE C-INDEX (Train/Val/Test/WHI)
#
# You can change ONLY:
#   MODEL_PATH  (to point to your saved stacking model)
#
# Data paths are derived from RUN_ID. Update RUN_ID or ROOT if folders are relocated.
# RUN_ID / ROOT if you later change where the prediction CSVs live.
# ==========================================================




# -----------------------
# 0) USER-EDITABLE MODEL PATH
# -----------------------
MODEL_PATH = r"./FHS60_pipeline/savedmodels/stacking_FHS60_FHS60_20260119171709/stacking_deepsurv_best_FHS60_20260119171709.keras"
# Example alt:
# MODEL_PATH = r"./FHS60_pipeline/savedmodels/stacking_FHS60_FHS60_20260119094456/stacking_deepsurv_final_FHS60_20260119094456_20260119120000.keras"


# -----------------------
# 1) HARDCODED DATA PATHS (edit RUN_ID/ROOT if you move runs)
# -----------------------
RUN_ID = "FHS60_20260119171709"
ROOT = Path(r"./FHS60_pipeline")

OOF_DIR = ROOT / "predictions" / f"FHS60_run_{RUN_ID}"
train_pred_path = OOF_DIR / "FHS60_preds_train.csv"
val_pred_path   = OOF_DIR / "FHS60_preds_val.csv"
test_pred_path  = OOF_DIR / "FHS60_preds_test.csv"
whi_pred_path   = OOF_DIR / "WHI_preds_external.csv"


# -----------------------
# 2) HELPERS
# -----------------------
def make_surv_from_df(df: pd.DataFrame):
    return Surv.from_dataframe("event", "time", df[["event", "time"]])

def cindex_sksurv(y_struct, scores):
    e = y_struct["event"].astype(bool)
    t = y_struct["time"].astype(float)
    s = np.asarray(scores).ravel()
    return float(concordance_index_censored(e, t, s)[0])


# -----------------------
# 3) LOAD PREDICTION CSVS + BUILD META FEATURES
# -----------------------
print("[INFO] Loading prediction CSVs...")
trainpreddf = pd.read_csv(train_pred_path)
valpreddf   = pd.read_csv(val_pred_path)
testpreddf  = pd.read_csv(test_pred_path)
whipreddf   = pd.read_csv(whi_pred_path)
print("[INFO] Shapes:", trainpreddf.shape, valpreddf.shape, testpreddf.shape, whipreddf.shape)

metacols = [c for c in trainpreddf.columns if c.startswith("pred_") and c.endswith("_scaled")]
if not metacols:
    raise ValueError("No meta-features found. Expected columns like 'pred_*_scaled'.")

# ensure consistency across splits
for df_name, df in [("val", valpreddf), ("test", testpreddf), ("whi", whipreddf)]:
    missing = [c for c in metacols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing meta-features: {missing}")

X_meta_train = trainpreddf[metacols].to_numpy(dtype=np.float32)
X_meta_val   = valpreddf[metacols].to_numpy(dtype=np.float32)
X_meta_test  = testpreddf[metacols].to_numpy(dtype=np.float32)
X_meta_whi   = whipreddf[metacols].to_numpy(dtype=np.float32)

y_train_struct = make_surv_from_df(trainpreddf)
y_val_struct   = make_surv_from_df(valpreddf)
y_test_struct  = make_surv_from_df(testpreddf)
y_whi_struct   = make_surv_from_df(whipreddf)

print("[INFO] Meta features used:", metacols)
print("[INFO] X_meta_train:", X_meta_train.shape)


# -----------------------
# 4) LOAD MODEL + PREDICT RISK SCORES
# -----------------------
print(f"[INFO] Loading model: {MODEL_PATH}")
# Keras recommends using tf.keras.models.load_model for models saved via model.save(...).
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

rs_train = model.predict(X_meta_train, batch_size=X_meta_train.shape[0], verbose=0)
rs_val   = model.predict(X_meta_val,   batch_size=X_meta_val.shape[0],   verbose=0)
rs_test  = model.predict(X_meta_test,  batch_size=X_meta_test.shape[0],  verbose=0)
rs_whi   = model.predict(X_meta_whi,   batch_size=X_meta_whi.shape[0],   verbose=0)

# -----------------------
# 5) COMPUTE C-INDEX
# -----------------------
c_train = cindex_sksurv(y_train_struct, rs_train)  # sksurv c-index expects risk scores (higher=more risk).
c_val   = cindex_sksurv(y_val_struct,   rs_val)
c_test  = cindex_sksurv(y_test_struct,  rs_test)
c_whi   = cindex_sksurv(y_whi_struct,   rs_whi)

print("\n=== CoxMLP Stacking C-index (loaded model) ===")
print(f"TRAIN C-index = {c_train:.4f}")
print(f"VAL   C-index = {c_val:.4f}")
print(f"TEST  C-index = {c_test:.4f}")
print(f"WHI   C-index = {c_whi:.4f}")
